**Making HTTP request to the Amazon**

In [3]:
import re
import time
import pandas as pd

!pip install selenium

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 8.8 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [4]:
def create_driver(headless=False):
    chrome_options = Options()

    if headless:
        # Selenium recommends setting headless through arguments
        chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")

    # If Chrome is installed in the normal location on Windows,
    # you usually do not need to set binary_location.

    driver = webdriver.Chrome(options=chrome_options)
    return driver

**PreProcessing Pipeline**

In [5]:
def check_block_page(driver):
    """
    Detect whether Amazon returned a CAPTCHA / robot check / blocked page.
    """
    page_source = driver.page_source.lower()
    title = driver.title.lower()

    block_signs = [
        "enter the characters you see below",
        "sorry, we just need to make sure you're not a robot",
        "captcha",
        "robot check"
    ]

    for sign in block_signs:
        if sign in page_source or sign in title:
            return True

    return False

In [6]:
def get_reviews_selenium(url, headless=False):
    driver = create_driver(headless=headless)
    reviews = []

    try:
        driver.get(url)

        time.sleep(20)

        if check_block_page(driver):
            print("Amazon blocked the request or showed a CAPTCHA page.")
            return reviews

        wait = WebDriverWait(driver, 10)
        wait.until(
            EC.presence_of_all_elements_located((By.XPATH, '//div[@data-hook="review"]'))
        )

        review_blocks = driver.find_elements(By.XPATH, '//div[@data-hook="review"]')

        for review in review_blocks:
            try:
                title = review.find_element(
                    By.XPATH, './/*[@data-hook="review-title"]'
                ).text.strip()
            except NoSuchElementException:
                title = None

            try:
                rating_text = review.find_element(
                    By.XPATH,
                    './/*[@data-hook="review-star-rating" or @data-hook="cmps-review-star-rating"]'
                ).get_attribute("textContent").strip()

                rating = float(rating_text.split()[0])
            except Exception:
                rating = None

            try:
                body = review.find_element(
                    By.XPATH, './/*[@data-hook="review-body"]'
                ).text.strip()
            except NoSuchElementException:
                body = None

            reviews.append({
                "title": title,
                "rating": rating,
                "review_text": body
            })

    except TimeoutException:
        print("Timed out waiting for Amazon reviews to load.")
    except Exception as e:
        print(f"Unexpected error: {e}")
    finally:
        driver.quit()

    return reviews

In [7]:
def clean_text(text):
    if not text:
        return ""

    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z0-9\\s]", "", text)
    text = text.lower()
    text = re.sub(r"\\s+", " ", text).strip()

    return text

In [8]:
def remove_duplicates(df):
    return df.drop_duplicates(subset=["review_text"])


def handle_missing_values(df):
    return df.dropna(subset=["review_text", "rating"])


def preprocess_data(reviews):
    df = pd.DataFrame(reviews)

    if df.empty:
        return df

    df = handle_missing_values(df)
    df["cleaned_review"] = df["review_text"].apply(clean_text)
    df = remove_duplicates(df)

    return df

In [9]:
def save_to_csv(df, filename="amazon_reviews.csv"):
    if df.empty:
        print("No data to save.")
        return

    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"Saved file as: {filename}")


if _name_ == "_main_":
    url = "https://www.amazon.com/product-reviews/B09B8V1FW3/"

    reviews = get_reviews_selenium(url, headless=False)
    print(f"Total raw reviews scraped: {len(reviews)}")

    df = preprocess_data(reviews)
    print(f"Total cleaned reviews: {len(df)}")

    if not df.empty:
        print(df.head())

    save_to_csv(df)

NameError: name '_name_' is not defined